In [ ]:
import pandas as pd
import os
import glob
from datetime import datetime

# 1. LOAD FILE LIST
# If you saved the CSV from the previous step:
# df = pd.read_csv("mea_file_locations.csv")

# OR, if you want to run it fresh right now:
root_search_path = "MEA_Analysis/AnalyzedData/CDKL5_T1"
#ANCHOR = root_search_path.split(os.sep)[-3]  # anything before project name
ANCHOR = "AnalyzedData"  # this is the folder that anchors the structure we want to parse
files = glob.glob(os.path.join(root_search_path, "**", "*network*.json"), recursive=True)
df = pd.DataFrame(files, columns=['full_path'])


def parse_path_metadata(path):
    """
    Extracts metadata from the standard path structure:
    .../AnalyzedData/Genotype/Date/ChipID/Network/RunID/WellID/...
    """
    try:
        parts = path.split(os.sep)
        
        # We anchor everything relative to "AnalyzedData" to be safe
        if ANCHOR in parts:
            idx = parts.index(ANCHOR)

            # Extract based on your specific directory depth
            project = parts[idx + 1]  # e.g., CDKL5_T1
            date_str = parts[idx + 2]  # e.g., 240531
            chip_id  = parts[idx + 3]  # e.g., M07420
            # parts[idx + 4] is usually "Network"
            run_id   = parts[idx + 5]  # e.g., 000052
            well_id  = parts[idx + 6]  # e.g., well005
            
            return pd.Series([project, date_str, chip_id, run_id, well_id])
        else:
            return pd.Series([None, None, None, None, None])
            
    except IndexError:
        return pd.Series([None, None, None, None, None])

# 2. APPLY EXTRACTION
print(" dissecting path names...")
metadata_cols = ["Project", "Date", "Chip_ID", "RunID", "Well"]
df[metadata_cols] = df['full_path'].apply(parse_path_metadata)
#puth full_path to last column
df = df[[col for col in df.columns if col != 'full_path'] + ['full_path']]


# 4. REVIEW
print(f"Successfully dissected {len(df)} paths.")
print("\nSnapshot of extracted metadata:")
print(df.head())

# 5. SAVE
#df.to_csv("/pscratch/sd/m/mpatil1/MEA_Analysis/IPNAnalysis/workbooks/mea_metadata_index_networks.csv", index=False)

# SEE the network json structure first to do the downstream analysis

In [ ]:
import json
import pandas as pd

# 1. Select the first file from our dissected dataframe
# We use .iloc[0] to grab the first row
sample_path = df.iloc[0]['full_path']
#print(f"Inspecting file: {sample_path}\n")
#sample_path = '/pscratch/sd/m/mpatil1/MEA_Analysis/AnalyzedData/CDKL5_T1/240531/M07420/Network/000052/well004/network_results.json'
# 2. Load the JSON data
with open(sample_path, 'r') as f:
    data = json.load(f)

# 3. Helper function to print the structure without flooding the screen
# This will show you the Keys and the Type of data inside (e.g., float, list of 6000 items)
def print_structure(d, indent=0):
    spacing = "  " * indent
    if isinstance(d, dict):
        for key, value in d.items():
            if isinstance(value, dict):
                print(f"{spacing}- {key}: (Nested Dictionary)")
                print_structure(value, indent + 1)
            elif isinstance(value, list):
                print(f"{spacing}- {key}: List [{len(value)} items]")
            else:
                print(f"{spacing}- {key}: {type(value).__name__} (e.g., {value})")
    else:
        print(f"{spacing} (Not a dictionary)")

# 4. Print the "Map" of your JSON
print("--- JSON FILE STRUCTURE ---")
print_structure(data)

# 5. Check for specific "Gold Mine" keys
# These are the standard keys usually found in MaxWell/MACS outputs.
# We try to print them specifically to see if they exist.
print("\n--- CHECKING FOR COMMON METRICS ---")
possible_keys = ['mean_firing_rate', 'mean_burst_rate', 'network_burst_frequency', 'burst_duration_mean']

found_metrics = {}
for key in possible_keys:
    # Check top level
    if key in data:
        found_metrics[key] = data[key]
    # Sometimes metrics are hidden inside a 'summary' or 'network' sub-dictionary
    elif 'summary' in data and key in data['summary']:
        found_metrics[key] = data['summary'][key]

print(f"Found specific metrics: {json.dumps(found_metrics, indent=2)}")

In [ ]:
import tqdm
import numpy as np
def extract_metrics(row):
    path = row['full_path']

    metrics = {}

    try:
        with open(path, "r") as f:
            data = json.load(f)

        # -----------------------------
        # Helper
        # -----------------------------
        def extract_block(block, prefix):
            if block is None:
                return

            m = block.get("metrics", {})
            events = block.get("events", [])

            # -------- Scalars --------
            metrics[f"{prefix}_count"] = m.get("count", np.nan)
            metrics[f"{prefix}_rate_hz"] = m.get("rate", np.nan)

            metrics[f"{prefix}_duration_mean"] = m.get("duration", {}).get("mean")
            metrics[f"{prefix}_duration_std"]  = m.get("duration", {}).get("std")
            metrics[f"{prefix}_duration_cv"]   = m.get("duration", {}).get("cv")

            metrics[f"{prefix}_ibi_mean"] = m.get("inter_event_interval", {}).get("mean")
            metrics[f"{prefix}_ibi_std"]  = m.get("inter_event_interval", {}).get("std")
            metrics[f"{prefix}_ibi_cv"]   = m.get("inter_event_interval", {}).get("cv")

            metrics[f"{prefix}_intensity_mean"] = m.get("intensity", {}).get("mean")

            metrics[f"{prefix}_spikes_mean"] = m.get("spikes_per_burst", {}).get("mean")
            metrics[f"{prefix}_participation_mean"] = m.get("participation", {}).get("mean")
            metrics[f"{prefix}_burst_peak_mean"] = m.get("burst_peak", {}).get("mean")
            metrics[f"{prefix}_peak_sync_mean"] = m.get("peak_synchrony", {}).get("mean")

            # -------- Distributions --------
            durations = []
            peaks = []
            intensities = []
            sync_energy = []
            fragments = []

            for ev in events:
                durations.append(ev.get("duration_s", np.nan))
                peaks.append(ev.get("peak_synchrony", np.nan))
                intensities.append(ev.get("total_spikes", np.nan))
                sync_energy.append(ev.get("synchrony_energy", np.nan))

                if "fragment_count" in ev:
                    fragments.append(ev["fragment_count"])

            metrics[f"{prefix}_durations"] = durations
            metrics[f"{prefix}_peaks"] = peaks
            metrics[f"{prefix}_intensity"] = intensities
            metrics[f"{prefix}_sync_energy"] = sync_energy

            if prefix == "nb":
                metrics[f"{prefix}_fragment_list"] = fragments
                metrics[f"{prefix}_fragment_mean"] = np.mean(fragments) if fragments else np.nan

        # -----------------------------
        # Apply
        # -----------------------------
        extract_block(data.get("burstlets"), "bl")
        extract_block(data.get("network_bursts"), "nb")
        extract_block(data.get("superbursts"), "sb")

        # -----------------------------
        # Diagnostics (DO NOT IGNORE THIS)
        # -----------------------------
        diag = data.get("diagnostics", {})
        for k, v in diag.items():
            metrics[f"diag_{k}"] = v

        metrics["n_units"] = data.get("n_units", np.nan)

    except Exception as e:
        print(f"[ERROR] {path}: {e}")

    return pd.Series(metrics)

# --- 3. RUN EXTRACTION ---
print("Starting unified extraction (Scalars + Lists)...")

# If you have tqdm installed, use progress_apply, otherwise use apply
if hasattr(tqdm, 'pandas'):
    df_metrics = df.progress_apply(extract_metrics, axis=1)
else:
    df_metrics = df.apply(extract_metrics, axis=1)

# Concatenate metadata with results
result_df = pd.concat([df, df_metrics], axis=1)



In [ ]:
import pandas as pd

ref_df = pd.read_excel("/pscratch/sd/m/mpatil1/Data/CDKL5_T1/CDKL5_T1_C1_reff_new.xlsx")
print(ref_df.columns)

# --- 2. TRANSFORMATION LOGIC ---

# Step A: Split the strings into actual Python lists
ref_df['Wells_List'] = ref_df['Wells_Recorded'].astype(str).str.split(',')
ref_df['Source_List'] = ref_df['Neuron Source'].astype(str).str.split(',')

# Step B: Explode the lists into rows (keeps alignment)
exploded_df = ref_df.explode(['Wells_List', 'Source_List'])

# Step C: Cleanup
exploded_df['Well'] = exploded_df['Wells_List'].str.strip()
exploded_df['NeuronType'] = exploded_df['Source_List'].str.strip()

# Step D: FIX — convert 1-based wells to 0-based and format
exploded_df['Well'] = (
    'well'
    + (exploded_df['Well'].astype(int) - 1)
      .astype(str)
      .str.zfill(3)
)

# Filter to keep only the clean columns
clean_ref_df = exploded_df[['Date', 'ID', 'Run #', 'Well', 'NeuronType', 'Assay', 'DIV']]

# --- 3. VIEW RESULTS ---
print("Transformation complete.")
print(f"Original rows: {len(ref_df)}")
print(f"Exploded rows: {len(clean_ref_df)}")
print("\nFirst 10 rows of clean metadata:")
print(clean_ref_df.head(10))

# Save if needed
# clean_ref_df.to_csv("clean_metadata_map.csv", index=False)

In [ ]:
result_df.rename(columns={"RunID": "Run #","Chip_ID": "ID"}, inplace=True)

In [ ]:
result_df['Date'] = result_df['Date'].astype(str)

In [ ]:
result_df['Run #'] = result_df['Run #'].astype(str).str.zfill(6)
clean_ref_df['Run #'] = clean_ref_df['Run #'].astype(str).str.zfill(6)
print(result_df['Well'].unique()[:10])
print(clean_ref_df['Well'].unique()[:10])

In [ ]:
# JSON-derived table must be unique
assert not result_df.duplicated(
    ['ID','Date','Run #','Well']
).any(), "Duplicate JSON-derived rows detected"

# Metadata table must be unique
assert not clean_ref_df.duplicated(
    ['ID','Date','Run #','Well']
).any(), "Duplicate metadata rows detected"

In [ ]:
merged_df = pd.merge(
    result_df,
    clean_ref_df,
    how='left',
    on=['ID', 'Run #', 'Well'],
    validate='one_to_one'
)

In [ ]:
#order by run #
merged_df.sort_values(by=['ID', 'Run #', 'Well'], inplace=True)

In [ ]:
merged_df.head()

In [ ]:
#save final merged table
merged_df.to_csv("/pscratch/sd/m/mpatil1/MEA_Analysis/AnalyzedData/CDKL5_T1/figures/final_merged_mea_data.csv", index=False)

In [ ]:
#split merged_df into two dataframes based on Assay type
network_assay_df = merged_df[merged_df['Assay'] == 'Network Today']
neuronal_assay_df = merged_df[merged_df['Assay'] == 'Neuronal Units 9']
#neuronal_assay_df = merged_df[merged_df['Assay'] == 'Neuronal unit']

In [ ]:
network_assay_df.head()

In [ ]:
# Supports:
# (ID, Well)       -> remove all DIVs for that well
# (ID, Well, DIV)  -> remove only that specific DIV
to_remove = [
    ('M07420', 'well005'),
    ('M07039', 'well004'),
    ('M07420', 'well002', 27),
    ('M07420', 'well004', 8),
]

mask_remove = pd.Series(False, index=network_assay_df.index)

for item in to_remove:
    if len(item) == 2:
        id_, well = item
        cond = (network_assay_df['ID'] == id_) & (network_assay_df['Well'] == well)
    elif len(item) == 3:
        id_, well, div = item
        cond = (
            (network_assay_df['ID'] == id_) &
            (network_assay_df['Well'] == well) &
            (network_assay_df['DIV'] == div)
        )
    else:
        raise ValueError(f"Invalid tuple format: {item}. Use (ID, Well) or (ID, Well, DIV).")

    mask_remove |= cond

removed_rows = int(mask_remove.sum())
network_assay_df = network_assay_df.loc[~mask_remove].copy()

print(f"Removed rows: {removed_rows}")
print(f"Remaining rows: {len(network_assay_df)}")


In [ ]:
palette = {'WT': '#4C72B0', 'HET': '#D55E00', 'KI': '#A63226',
               'MxWT': '#4C72B0', 'FxHET': '#D55E00', 'MxHEMI': '#A63226'}
markers = ['o','s','d','*','^']
colors=[]
unique_genotypes = network_assay_df['NeuronType'].unique()
for i, genotype in enumerate(unique_genotypes):
    colors.append(palette[genotype])
order =['MxWT','FxHET','MxHEMI']


In [ ]:
import sys
sys.path.append("/pscratch/sd/m/mpatil1/MEA_Analysis/IPNAnalysis")
import meaplotter
import importlib
importlib.reload(meaplotter)
viz = meaplotter.MEAPlotter(
    group_order=order,
    palette=palette
)

In [ ]:
network_assay_df['NeuronType'].unique()

In [ ]:
# Remove all rows where DIV == 5 from the network dataframe
before_rows = len(network_assay_df)
network_assay_df = network_assay_df.loc[network_assay_df['DIV'] != 5].copy()
removed_rows = before_rows - len(network_assay_df)

print(f"Removed DIV 5 rows: {removed_rows}")
print(f"Remaining rows: {len(network_assay_df)}")

In [ ]:
columns_to_plot = [ 'bl_burst_peak_mean', 'bl_count', 'bl_duration_cv', 'bl_duration_mean',
       'bl_duration_std', 'bl_durations', 'bl_ibi_cv', 'bl_ibi_mean',
       'bl_ibi_std', 'bl_intensity', 'bl_intensity_mean',
       'bl_participation_mean', 'bl_peak_sync_mean', 'bl_peaks', 'bl_rate_hz',
       'bl_spikes_mean', 'bl_sync_energy', 'diag_adaptive_bin_ms',
       'diag_baseline_value', 'diag_biological_isi_s',
       'diag_burstlet_merge_gap_s', 'diag_merge_floor', 'diag_n_units',
       'diag_network_merge_gap_s', 'diag_sigma_fast_bins',
       'diag_sigma_slow_bins', 'diag_smoothing_sigma_fast_bins',
       'diag_smoothing_sigma_slow_bins', 'diag_spread_mad',
       'diag_threshold_value', 'n_units', 'nb_burst_peak_mean', 'nb_count',
       'nb_duration_cv', 'nb_duration_mean', 'nb_duration_std', 'nb_durations',
       'nb_fragment_list', 'nb_fragment_mean', 'nb_ibi_cv', 'nb_ibi_mean',
       'nb_ibi_std', 'nb_intensity', 'nb_intensity_mean',
       'nb_participation_mean', 'nb_peak_sync_mean', 'nb_peaks', 'nb_rate_hz',
       'nb_spikes_mean', 'nb_sync_energy', 'sb_burst_peak_mean', 'sb_count',
       'sb_duration_cv', 'sb_duration_mean', 'sb_duration_std', 'sb_durations',
       'sb_ibi_cv', 'sb_ibi_mean', 'sb_ibi_std', 'sb_intensity',
       'sb_intensity_mean', 'sb_participation_mean', 'sb_peak_sync_mean',
       'sb_peaks', 'sb_rate_hz', 'sb_spikes_mean', 'sb_sync_energy'
       ]
PATH = "/pscratch/sd/m/mpatil1/MEA_Analysis/AnalyzedData/CDKL5_T1/figures"
for col in columns_to_plot:
  
  try:
      ax = viz.plot_line_sem_by_div(
          network_assay_df,
          div_col="DIV",
          group_col="NeuronType",
          y=col,
          title=f"{col}",
          show_legend=False,
          fig_width = 2.5,
          aspect = 0.8
      )
      ax.get_figure().savefig(f"{PATH}/{col}.svg", dpi=300,format="svg") 
      stat_df = viz.calculate_stats_by_div_welch(network_assay_df, div_col="DIV", group_col="NeuronType", y=col, group_order=order)
      stat_df.to_csv(f"{PATH}/{col}_stats.csv", index=False)
  except Exception as e:
      print(f"Could not plot {col}: {e}")


In [ ]:













import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy.stats import sem, ttest_ind, levene
from itertools import combinations

def analyze_organoid_pairwise_final(
    df,
    value_col,
    order=('Healthy', 'E198K', 'E200K'),
    palette={'Healthy': 'black', 'E198K': '#ff7f0e', 'E200K': 'cornflowerblue'},
    feature_label=None,
    save_path=None
):
    """
    Standardized HD-MEA Analysis for Organoids:
    - Auto-selects Student/Welch T-test per comparison.
    - Publication-quality brackets (no messy text on plots).
    - Robust error handling for NaN/Zero-variance data.
    """
    
    # 1. DATA PREP & CLEANING
    sub = df[['DIV', 'NeuronType', value_col]].copy().dropna(subset=[value_col])
    
    # Handle entirely empty columns
    if sub.empty:
        print(f"[SKIP] {value_col}: No data points found.")
        return

    divs = sorted(sub['DIV'].unique())

    # 2. SUMMARY TABLE (Mean, SEM, N)
    summary = sub.groupby(['DIV', 'NeuronType']).agg(
        mean=(value_col, 'mean'),
        sem=(value_col, lambda x: sem(x, nan_policy='omit')),
        n=(value_col, 'count')
    ).reindex(order, level='NeuronType').reset_index()

    # 3. PAIRWISE STATS WITH AUTO-SELECTION (Student vs Welch)
    stats_list = []
    pairs = list(combinations(order, 2))

    def get_stars(p):
        if p < 0.001: return '***'
        if p < 0.01: return '**'
        if p < 0.05: return '*'
        return None

    for div in divs:
        day_data = sub[sub['DIV'] == div]
        for g1, g2 in pairs:
            v1 = day_data[day_data['NeuronType'] == g1][value_col].values
            v2 = day_data[day_data['NeuronType'] == g2][value_col].values

            if len(v1) >= 2 and len(v2) >= 2:
                # Levene's test to check variance equality
                _, p_lev = levene(v1, v2)
                equal_var = p_lev > 0.05
                test_type = "Student" if equal_var else "Welch"

                # Run appropriate T-test
                _, p = ttest_ind(v1, v2, equal_var=equal_var, nan_policy='omit')
                
                stats_list.append({
                    'DIV': div, 'G1': g1, 'G2': g2, 'Comparison': f"{g1} vs {g2}",
                    'p_raw': p, 'test_used': test_type, 'sig': get_stars(p)
                })
            else:
                stats_list.append({
                    'DIV': div, 'G1': g1, 'G2': g2, 'Comparison': f"{g1} vs {g2}",
                    'p_raw': np.nan, 'test_used': 'N<2', 'sig': None
                })

    stats_df = pd.DataFrame(stats_list)

    # 4. PLOTTING WITH ROBUST LIMITS
    fig, ax = plt.subplots(figsize=(10, 6))
    DIV_SPACING, GENO_SPACING = 2.0, 0.3
    div_to_x = {d: i * DIV_SPACING for i, d in enumerate(divs)}
    offsets = {g: (i - (len(order)-1)/2) * GENO_SPACING for i, g in enumerate(order)}

    # --- CRITICAL: Robust Y-Limit Logic to prevent NaN/Inf Errors ---
    yvals = sub[value_col].dropna().values
    yvals = yvals[np.isfinite(yvals)]
    
    if len(yvals) == 0:
        print(f"[SKIP] {value_col}: No finite values to plot.")
        plt.close(fig)
        return

    y_l, y_h = np.nanpercentile(yvals, 1), np.nanpercentile(yvals, 99)
    yr = (y_h - y_l)
    
    # If data is flat, create a dummy range
    if yr == 0:
        yr = abs(y_h) * 0.1 if y_h != 0 else 1.0
        
    if not np.isfinite(y_l) or not np.isfinite(y_h):
        print(f"[SKIP] {value_col}: Calculation resulted in non-finite limits.")
        plt.close(fig)
        return

    ax.set_ylim(y_l - 0.15 * yr, y_h + 0.8 * yr)

    # 5. DRAW DATA
    for nt in order:
        d_nt = sub[sub['NeuronType'] == nt]
        s_nt = summary[summary['NeuronType'] == nt]
        
        # Jittered Raw Points
        x_raw = d_nt['DIV'].map(div_to_x) + offsets[nt] + (np.random.rand(len(d_nt))-0.5)*0.1
        ax.scatter(x_raw, d_nt[value_col], color=palette[nt], alpha=0.15, s=25, zorder=1)
        
        # Mean & SEM Line
        x_m = s_nt['DIV'].map(div_to_x) + offsets[nt]
        ax.plot(x_m, s_nt['mean'], '-o', color=palette[nt], label=nt, lw=2.5, zorder=3)
        ax.fill_between(x_m, s_nt['mean']-s_nt['sem'], s_nt['mean']+s_nt['sem'], 
                        color=palette[nt], alpha=0.1, zorder=2)

    # 6. DRAW SIGNIFICANCE BRACKETS
    for div in divs:
        div_sig = stats_df[(stats_df['DIV'] == div) & (stats_df['sig'].notna())]
        
        # Determine starting height for brackets at this DIV
        y_max_at_div = summary[summary['DIV'] == div].apply(lambda x: x['mean'] + x['sem'], axis=1).max()
        # Ensure brackets are at least above the local data points
        y_start = max(y_max_at_div + 0.05 * yr, y_l + 0.1 * yr)

        for i, (_, row) in enumerate(div_sig.iterrows()):
            x1 = div_to_x[div] + offsets[row['G1']]
            x2 = div_to_x[div] + offsets[row['G2']]
            
            # Increment Y height for multiple brackets
            y = y_start + (i * 0.15 * yr)
            h = 0.03 * yr # bracket tick height
            
            # Drawing
            ax.plot([x1, x1, x2, x2], [y-h, y, y, y-h], lw=1.2, color='#444444')
            ax.text((x1+x2)/2, y + 0.01*yr, row['sig'], ha='center', va='bottom', 
                    fontsize=12, fontweight='bold', color='black')

    # 7. FINAL POLISH
    ax.set_xticks([div_to_x[d] for d in divs])
    ax.set_xticklabels(divs)
    ax.set_xlabel('Days In Vitro (DIV)', fontweight='bold')
    ax.set_ylabel(feature_label or value_col, fontweight='bold')
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1,1))
    ax.spines[['top', 'right']].set_visible(False)
    
    # 8. CONSOLE OUTPUT
    print(f"\n{'='*60}\nFEATURE: {value_col}\n{'='*60}")
    print("\n[SUMMARY TABLE]")
    print(summary[['DIV', 'NeuronType', 'n', 'mean', 'sem']].to_string(index=False))
    print("\n[STATISTICAL LOG: Student/Welch Selection]")
    print(stats_df[['DIV', 'Comparison', 'test_used', 'p_raw', 'sig']].dropna(subset=['p_raw']).to_string(index=False))

    plt.tight_layout()
    if save_path: 
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    #plt.show()
    return fig


In [ ]:
cols_to_plot = [ 'nb_count',
       'nb_rate_hz', 'nb_duration_mean_s', 'nb_ibi_mean_s','nb_fragment_count_mean',
       'nb_spikes_per_burst_mean', 'nb_participation_mean',
     'sb_count', 'sb_rate_hz',
       'sb_duration_mean_s', 'sb_ibi_mean_s', 'sb_spikes_per_burst_mean',
       'sb_participation_mean', 'bl_count',
       'bl_rate_hz', 'bl_duration_mean_s', 'bl_ibi_mean_s',
       'bl_spikes_per_burst_mean', 'bl_participation_mean',
        'num_units'
       ]
for col in cols_to_plot:
    analyze_organoid_pairwise_final(
        neuronal_assay_df,
        value_col=col,
        feature_label=col.replace('_', ' ').title(),
        order=(  'CSB3 Healthy', 'CSB3 E198K','CSB3 E200K'),
        palette={ 'CSB3 Healthy': 'black','CSB3 E198K': '#ff7f0e', 'CSB3 E200K': 'cornflowerblue'},
        #save_path=f"/pscratch/sd/m/mpatil1/MEA_Analysis/IPNAnalysis/workbooks/plots/{col}_lineplot.svg"
    )

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

def export_all_to_pdf(df, cols, filename="MEA_Analysis_Report.pdf"):
    """
    Runs the pairwise analysis for all columns and saves them into a single PDF.
    """
    # Initialize the PDF document
    with PdfPages(filename) as pdf:
        for col in cols:
            print(f"Processing: {col}...")
            
            # 1. Generate the plot using our robust function
            # Note: We need to modify the function slightly to RETURN the fig
            fig = analyze_organoid_pairwise_final(
                df,
                value_col=col,
                feature_label=col.replace('_', ' ').title(),
                order=('Healthy', 'E198K', 'E200K'),
                palette={'Healthy': 'black', 'E198K': '#ff7f0e', 'E200K': 'cornflowerblue'}
            )
            
            # 2. If a figure was successfully created, save it to the PDF
            if fig is not None:
                pdf.savefig(fig)
                plt.close(fig) # Memory management: very important for 20+ plots
            
    print(f"\nSuccessfully exported all {len(cols)} features to {filename}")

# --- QUICK TWEAK TO YOUR FUNCTION ---
# Ensure your 'analyze_organoid_pairwise_final' ends with:
# return fig 
# instead of plt.show()

In [ ]:
neuronal_assay_df.columns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import sem, ttest_ind, levene
from itertools import combinations
from matplotlib.backends.backend_pdf import PdfPages

def analyze_organoid_pairwise_pdf_version(
    df, 
    value_col, 
    order=['Healthy','Healthy_PPP2RSD+', 'E198K','E198K_PPP2RSD+', 'E200K',
       'E200K_PPP2RSD+'],
    #order=['CSB3 Healthy', 'CSB3 E198K', 'CSB3 E200K'],
    #palette={'CSB3 Healthy': '#444444', 'CSB3 E198K': '#d98b5f', 'CSB3 E200K': '#8ca2ba'},
    palette= {'Healthy_PPP2RSD+': 'purple', 'Healthy': 'black', 'E198K_PPP2RSD+': 'purple', 'E198K': 'black',
       'E200K_PPP2RSD+': 'purple', 'E200K': 'black'},
    feature_label=None
):
    # 1. DATA PREP
    sub = df[df['NeuronType'].isin(order)].copy()
    sub = sub[['DIV', 'NeuronType', value_col]].dropna(subset=[value_col])
    
    if sub.empty:
        return None, None

    divs = sorted(sub['DIV'].unique())
    
    # 2. SUMMARY TABLE
    summary = sub.groupby(['DIV', 'NeuronType']).agg(
        mean=(value_col, 'mean'),
        sem=(value_col, lambda x: sem(x, nan_policy='omit')),
        n=(value_col, 'count')
    ).reindex(pd.MultiIndex.from_product([divs, order], names=['DIV', 'NeuronType'])).reset_index()

    # 3. PAIRWISE STATS
    stats_list = []
    pairs = list(combinations(order, 2))

    def get_stars(p):
        if p < 0.0001: return '****'
        if p < 0.001: return '***'
        if p < 0.01: return '**'
        if p < 0.05: return '*'
        return 'ns'

    for div in divs:
        day_data = sub[sub['DIV'] == div]
        for g1, g2 in pairs:
            v1 = day_data[day_data['NeuronType'] == g1][value_col].values
            v2 = day_data[day_data['NeuronType'] == g2][value_col].values
            if len(v1) >= 2 and len(v2) >= 2:
                _, p_lev = levene(v1, v2)
                equal_var = p_lev > 0.05
                test_type = "Student" if equal_var else "Welch"
                _, p = ttest_ind(v1, v2, equal_var=equal_var, nan_policy='omit')
                stats_list.append({'DIV': div, 'G1': g1, 'G2': g2, 'Comparison': f"{g1} vs {g2}",
                                   'p_raw': p, 'test_used': test_type, 'sig': get_stars(p)})
    stats_df = pd.DataFrame(stats_list)

    # 4. FIGURE: GROUPED BARS ACROSS DIVs
    fig_plot, ax = plt.subplots(figsize=(max(6, len(divs)*3), 6))
    
    n_groups = len(divs)
    n_bars = len(order)
    bar_width = 0.25
    group_spacing = 1.2  # Space between DIV clusters
    
    # Generate X positions
    indices = np.arange(n_groups) * group_spacing
    
    for i, nt in enumerate(order):
        # Filter summary for this genotype across all DIVs
        nt_summary = summary[summary['NeuronType'] == nt]
        x_offsets = indices + (i * bar_width)
        
        # Plot Bars
        ax.bar(x_offsets, nt_summary['mean'], width=bar_width, color=palette[nt], 
               alpha=0.6, edgecolor='black', linewidth=1, label=nt if i == 0 else "")
        
        # Plot Error Bars
        ax.errorbar(x_offsets, nt_summary['mean'], yerr=nt_summary['sem'], 
                    fmt='none', c='black', capsize=3, lw=1)
        
        # Plot Jittered Scatter for each DIV
        for j, div in enumerate(divs):
            points = sub[(sub['DIV'] == div) & (sub['NeuronType'] == nt)][value_col]
            jitter = (np.random.rand(len(points)) - 0.5) * (bar_width * 0.6)
            ax.scatter(np.full(len(points), x_offsets[j]) + jitter, points, 
                       color=palette[nt], edgecolor='black', alpha=0.4, s=20, zorder=3)

    # 5. STATISTICAL BRACKETS (Grouped per DIV)
    y_max_all = sub[value_col].max()
    yr = y_max_all * 0.1
    
    for j, div in enumerate(divs):
        div_x_start = indices[j]
        # Comparison 1: Healthy vs E198K
        comp1 = stats_df[(stats_df['DIV'] == div) & (stats_df['G1'] == order[0]) & (stats_df['G2'] == order[1])]
        if not comp1.empty and comp1['sig'].values[0] != 'ns':
            sig = comp1['sig'].values[0]
            x1, x2 = div_x_start, div_x_start + bar_width
            y = y_max_all + yr
            ax.plot([x1, x1, x2, x2], [y-(yr*0.2), y, y, y-(yr*0.2)], color='black', lw=1)
            ax.text((x1+x2)/2, y, sig, ha='center', va='bottom', fontsize=9, fontweight='bold')
            
        # Comparison 2: Healthy vs E200K
        comp2 = stats_df[(stats_df['DIV'] == div) & (stats_df['G1'] == order[0]) & (stats_df['G2'] == order[2])]
        if not comp2.empty and comp2['sig'].values[0] != 'ns':
            sig = comp2['sig'].values[0]
            x1, x2 = div_x_start, div_x_start + (2 * bar_width)
            y = y_max_all + (yr * 2.5)
            ax.plot([x1, x1, x2, x2], [y-(yr*0.2), y, y, y-(yr*0.2)], color='black', lw=1)
            ax.text((x1+x2)/2, y, sig, ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Formatting
    ax.set_xticks(indices + bar_width)
    ax.set_xticklabels([f"DIV {d}" for d in divs], fontweight='bold')
    ax.set_ylabel(feature_label or value_col, fontweight='bold')
    ax.set_title(feature_label, fontweight='bold', fontsize=14, pad=20)
    ax.spines[['top', 'right']].set_visible(False)
    
    # Legend with cleaned names
    handles, labels = ax.get_legend_handles_labels()
    ax.legend([plt.Rectangle((0,0),1,1, color=palette[nt]) for nt in order], 
              [n.replace('CSB3 ', '') for n in order], frameon=False, loc='upper left', bbox_to_anchor=(1,1))

    plt.tight_layout()

    # 6. STATS TABLE PAGE
    fig_stats, ax_st = plt.subplots(figsize=(10, 8))
    ax_st.axis('off')
    if not stats_df.empty:
        display_stats = stats_df[['DIV', 'Comparison', 'p_raw', 'sig']].copy()
        display_stats['p_raw'] = display_stats['p_raw'].apply(lambda x: f"{x:.4e}" if pd.notnull(x) else "N/A")
        the_table = ax_st.table(cellText=display_stats.values, colLabels=display_stats.columns, loc='center', cellLoc='center')
        the_table.auto_set_font_size(False); the_table.set_fontsize(8); the_table.scale(1.2, 1.2)
    
    return fig_plot, fig_stats

def export_mea_analysis_to_pdf(df, cols,order,palette, output_name="MEA_Grouped_Report.pdf"):
    with PdfPages(output_name) as pdf:
        for col in cols:
            print(f"Plotting {col} across all DIVs...")
            label = col.replace('nb_', '').replace('_', ' ').title()
            fig_p, fig_s = analyze_organoid_pairwise_pdf_version(df, col, order=order, palette=palette, feature_label=label)
            if fig_p is not None:
                pdf.savefig(fig_p)
                pdf.savefig(fig_s)
                plt.close(fig_p)
                plt.close(fig_s)
    print(f"\nReport complete: {output_name}")

# --- EXECUTION ---
cols_to_plot = [
'num_units', 'bl_count', 'nb_count',
       'nb_rate_hz', 'nb_duration_mean_s', 'nb_ibi_mean_s',
       'nb_spikes_per_burst_mean', 'nb_participation_mean',
       'nb_burst_peak_mean','nb_peak_synchrony_mean',
       'nb_fragment_count_mean'
]

export_mea_analysis_to_pdf(neuronal_assay_df, cols_to_plot,  order=(  'CSB3 Healthy', 'CSB3 E198K','CSB3 E200K'),
        palette={ 'CSB3 Healthy': 'black','CSB3 E198K': '#ff7f0e', 'CSB3 E200K': 'cornflowerblue'},
                           output_name='/mnt/Vol20tb1/user_workspaces/shruti/MEA_Analysis/MEA_Analysis_V2/MEA_Analysis/AnalyzedData/JGA_CSB3_010825_VD/MEA_Graphs_Grouped.pdf')

In [ ]:
export_mea_analysis_to_pdf(neuronal_assay_df[neuronal_assay_df['DIV'] == 19], cols_to_plot,  order=(  'CSB3 Healthy', 'CSB3 E198K','CSB3 E200K'),
        palette={ 'CSB3 Healthy': 'black','CSB3 E198K': '#ff7f0e', 'CSB3 E200K': 'cornflowerblue'},
                           output_name='/mnt/Vol20tb1/user_workspaces/shruti/MEA_Analysis/MEA_Analysis_V2/MEA_Analysis/AnalyzedData/JGA_CSB3_010825_VD/MEA_Graphs_DIV19.pdf')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import sem, ttest_ind
from itertools import combinations
from matplotlib.backends.backend_pdf import PdfPages

def analyze_organoid_pairwise_pdf_version(
    df, 
    value_col, 
    group_col='NeuronType',
    order=['Healthy','Healthy_PPP2RSD+', 'E198K','E198K_PPP2RSD+', 'E200K','E200K_PPP2RSD+'],
    palette= {'Healthy': 'black', 'Healthy_PPP2RSD+': 'purple', 
              'E198K': 'black', 'E198K_PPP2RSD+': 'purple', 
              'E200K': 'black', 'E200K_PPP2RSD+': 'purple'},
    feature_label=None
):
    # 1. DATA PREP
    sub = df[df[group_col].isin(order)].copy()
    sub = sub.dropna(subset=[value_col])
    if sub.empty: 
        print(f"No data for {value_col}")
        return None, None, None

    divs = sorted(sub['DIV'].unique())
    
    # Statistics Helpers
    def get_stars(p):
        if p < 0.0001: return '****'
        if p < 0.001: return '***'
        if p < 0.01: return '**'
        if p < 0.05: return '*'
        return 'ns'

    def run_welch_stats(data_subset, g_col, v_col, pairs):
        results = []
        for g1, g2 in pairs:
            v1 = data_subset[data_subset[g_col] == g1][v_col].values
            v2 = data_subset[data_subset[g_col] == g2][v_col].values
            if len(v1) >= 2 and len(v2) >= 2:
                _, p = ttest_ind(v1, v2, equal_var=False, nan_policy='omit')
                results.append({'G1': g1, 'G2': g2, 'Comparison': f"{g1} vs {g2}", 
                                'p_raw': p, 'test_used': "Welch's T", 'sig': get_stars(p)})
        return pd.DataFrame(results)

    # 2. RUN STATISTICS
    pairs = list(combinations(order, 2))
    div_stats_list = []
    for div in divs:
        d_stats = run_welch_stats(sub[sub['DIV'] == div], group_col, value_col, pairs)
        if not d_stats.empty:
            d_stats['DIV'] = div
            div_stats_list.append(d_stats)
    
    stats_df = pd.concat(div_stats_list) if div_stats_list else pd.DataFrame()
    agg_stats_df = run_welch_stats(sub, group_col, value_col, pairs)

    # 3. FIGURE 1: GROUPED BARS ACROSS DIVs
    summary = sub.groupby(['DIV', group_col])[value_col].agg(['mean', lambda x: sem(x, nan_policy='omit')])
    summary.columns = ['mean', 'sem']
    summary = summary.reset_index()

    fig_div, ax = plt.subplots(figsize=(max(12, len(divs)*4.5), 8))
    group_spacing = 3.0 
    bar_width = 0.32
    indices = np.arange(len(divs)) * group_spacing
    
    y_max_global = sub[value_col].max()
    yr = y_max_global * 0.12

    for i, nt in enumerate(order):
        nt_summary = summary[summary[group_col] == nt].set_index('DIV').reindex(divs).reset_index()
        x_off = indices + (i * bar_width)
        
        ax.bar(x_off, nt_summary['mean'], width=bar_width, color=palette.get(nt, 'gray'), 
               alpha=0.6, edgecolor='black', linewidth=1.2)
        ax.errorbar(x_off, nt_summary['mean'], yerr=nt_summary['sem'], fmt='none', c='black', capsize=3)
        
        # GENOTYPE SUB-LABELS
        if i % 2 == 0:
            geno_text = nt.replace('_PPP2RSD+', '')
            for j, idx in enumerate(indices):
                ax.text(idx + (i * bar_width) + bar_width/2, -y_max_global*0.06, 
                        geno_text, ha='center', va='top', fontsize=9, fontweight='bold')

        for j, div in enumerate(divs):
            pts = sub[(sub['DIV'] == div) & (sub[group_col] == nt)][value_col]
            jitter = (np.random.rand(len(pts)) - 0.5) * (bar_width * 0.5)
            ax.scatter(np.full(len(pts), x_off[j]) + jitter, pts, color=palette.get(nt, 'gray'), 
                       edgecolor='white', alpha=0.4, s=15, zorder=3)

    # Brackets
    for j, div in enumerate(divs):
        div_x = indices[j]
        d_stats = stats_df[stats_df['DIV'] == div] if not stats_df.empty else pd.DataFrame()
        if d_stats.empty: continue

        def draw_b(idx1, idx2, y_lvl):
            p_stats = d_stats[((d_stats['G1'] == order[idx1]) & (d_stats['G2'] == order[idx2]))]
            if not p_stats.empty and p_stats['sig'].values[0] != 'ns':
                sig = p_stats['sig'].values[0]
                x1, x2 = div_x + (idx1 * bar_width), div_x + (idx2 * bar_width)
                y = y_max_global + (y_lvl * yr)
                ax.plot([x1, x1, x2, x2], [y-(yr*0.2), y, y, y-(yr*0.2)], color='black', lw=1)
                ax.text((x1+x2)/2, y, sig, ha='center', va='bottom', fontsize=8, fontweight='bold')

        draw_b(0, 1, 0.5); draw_b(2, 3, 0.5); draw_b(4, 5, 0.5)
        draw_b(0, 2, 1.8); draw_b(2, 4, 1.8); draw_b(0, 4, 3.8)
        draw_b(1, 3, 2.8); draw_b(3, 5, 2.8); draw_b(1, 5, 4.8)

    ax.set_xticks(indices + (bar_width * 2.5))
    ax.set_xticklabels([f"DIV {int(d)}" for d in divs], fontweight='bold', fontsize=12)
    # Correct way to add padding to tick labels
    ax.tick_params(axis='x', which='major', pad=35) 
    
    ax.set_ylabel(feature_label, fontweight='bold')
    ax.set_title(f"{feature_label} per DIV (Paired NT/AAV Comparison)", fontweight='bold', pad=35)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, y_max_global + (yr * 7))
    
    legend_elements = [plt.Rectangle((0,0),1,1, color='black', alpha=0.6, label='NT (No Treatment)'),
                       plt.Rectangle((0,0),1,1, color='purple', alpha=0.6, label='AAV (PPP2RSD+)')]
    ax.legend(handles=legend_elements, frameon=False, loc='upper left', bbox_to_anchor=(1,1))

    # 4. FIGURE 2: AGGREGATED
    agg_summary = sub.groupby(group_col)[value_col].agg(['mean', lambda x: sem(x, nan_policy='omit')])
    agg_summary.columns = ['mean', 'sem']
    agg_summary = agg_summary.reindex(order).reset_index()

    fig_agg, ax_agg = plt.subplots(figsize=(7, 7))
    for i, nt in enumerate(order):
        row = agg_summary.iloc[i]
        pts = sub[sub[group_col] == nt][value_col]
        ax_agg.bar(i, row['mean'], width=0.7, color=palette.get(nt, 'gray'), alpha=0.6, edgecolor='black')
        ax_agg.errorbar(i, row['mean'], yerr=row['sem'], fmt='none', c='black', capsize=5)
        ax_agg.scatter(np.full(len(pts), i) + (np.random.rand(len(pts))-0.5)*0.4, pts, 
                       color=palette.get(nt, 'gray'), edgecolor='white', alpha=0.4, s=20, zorder=3)

    for idx1, idx2, y_lvl in [(0,1,0.5), (2,3,0.5), (4,5,0.5), (0,2,1.8), (2,4,1.8), (0,4,3.8)]:
        sig_row = agg_stats_df[(agg_stats_df['G1'] == order[idx1]) & (agg_stats_df['G2'] == order[idx2])]
        if not sig_row.empty and sig_row['sig'].values[0] != 'ns':
            y = y_max_global + (yr * y_lvl)
            ax_agg.plot([idx1, idx1, idx2, idx2], [y-(yr*0.2), y, y, y-(yr*0.2)], color='black', lw=1)
            ax_agg.text((idx1+idx2)/2, y, sig_row['sig'].values[0], ha='center', va='bottom', fontweight='bold')

    ax_agg.set_xticks(range(len(order)))
    ax_agg.set_xticklabels([n.replace('_PPP2RSD+', '+') for n in order], rotation=45, fontweight='bold')
    ax_agg.set_ylabel(feature_label, fontweight='bold')
    ax_agg.set_title(f"{feature_label} (Aggregated)", fontweight='bold', pad=25)
    ax_agg.spines[['top', 'right']].set_visible(False)
    ax_agg.set_ylim(0, y_max_global + (yr * 7))

    # 5. FIGURE 3: STATS TABLE
    fig_tbl, ax_tbl = plt.subplots(figsize=(10, 10))
    ax_tbl.axis('off')
    if not stats_df.empty:
        agg_tbl = agg_stats_df.copy(); agg_tbl['DIV'] = 'AGGREGATED'
        full_stats = pd.concat([stats_df, agg_tbl])[['DIV', 'Comparison', 'p_raw', 'sig']]
        full_stats['p_raw'] = full_stats['p_raw'].apply(lambda x: f"{x:.4e}" if pd.notnull(x) else "N/A")
        table = ax_tbl.table(cellText=full_stats.values, colLabels=full_stats.columns, loc='center', cellLoc='center')
        table.auto_set_font_size(False); table.set_fontsize(7); table.scale(1.1, 1.1)
        #ax_tbl.set_title(f"Detailed Statistics Table: {feature_label}", fontweight='bold')

    plt.tight_layout()
    return fig_div, fig_agg, fig_tbl

def export_mea_analysis_to_pdf(df, cols, output_name="Report.pdf"):
    with PdfPages(output_name) as pdf:
        for col in cols:
            if col not in df.columns: continue
            print(f"Processing Analysis: {col}")
            label = col.replace('nb_', '').replace('_', ' ').title()
            f1, f2, f3 = analyze_organoid_pairwise_pdf_version(df, col, feature_label=label)
            if f1 is not None:
                pdf.savefig(f1); pdf.savefig(f2); pdf.savefig(f3)
                plt.close(f1); plt.close(f2); plt.close(f3)
    print(f"\nCompleted! PDF saved: {output_name}")

# --- EXECUTION ---
cols_to_plot = ['nb_count', 'nb_rate_hz', 'nb_duration_mean_s', 'nb_spikes_per_burst_mean', 'num_units', 'bl_count', 'nb_burst_peak_mean']
export_mea_analysis_to_pdf(neuronal_assay_df, cols_to_plot, 
                           output_name='/mnt/Vol20tb1/user_workspaces/shruti/MEA_Analysis/MEA_Analysis_V2/MEA_Analysis/AnalyzedData/JGA_COB7_080825_VD/MEA_Graphs_Final_Paired_Labeled.pdf')

In [ ]:
neuronal_assay_df.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import sem, ttest_ind
from itertools import combinations
from matplotlib.backends.backend_pdf import PdfPages

def analyze_organoid_pairwise_pdf_version(
    df, 
    value_col, 
    group_col='NeuronType',
    order=['CSB3 Healthy', 'CSB3 E198K', 'CSB3 E200K'],
    palette={'CSB3 Healthy': '#444444', 'CSB3 E198K': '#d98b5f', 'CSB3 E200K': '#8ca2ba'},
    feature_label=None
):
    # 1. DATA PREP
    # Filter for genotypes in order and drop NaNs
    sub = df[df[group_col].isin(order)].copy()
    sub = sub.dropna(subset=[value_col])
    
    if sub.empty:
        print(f"No data for {value_col}")
        return None, None, None

    divs = sorted(sub['DIV'].unique())
    
    # --- STATISTICS UTILITY (Strictly Welch's) ---
    def get_stars(p):
        if p < 0.0001: return '****'
        if p < 0.001: return '***'
        if p < 0.01: return '**'
        if p < 0.05: return '*'
        return 'ns'

    def run_welch_stats(data_subset, g_col, v_col, pairs):
        results = []
        for g1, g2 in pairs:
            v1 = data_subset[data_subset[g_col] == g1][v_col].values
            v2 = data_subset[data_subset[g_col] == g2][v_col].values
            if len(v1) >= 2 and len(v2) >= 2:
                # Independent t-test with Welch's correction
                _, p = ttest_ind(v1, v2, equal_var=False, nan_policy='omit')
                results.append({
                    'G1': g1, 'G2': g2, 'Comparison': f"{g1} vs {g2}", 
                    'p_raw': p, 'test_used': "Welch's T", 'sig': get_stars(p)
                })
        return pd.DataFrame(results)

    # 2. RUN STATISTICS
    pairs = list(combinations(order, 2))
    div_stats_list = []
    for div in divs:
        d_stats = run_welch_stats(sub[sub['DIV'] == div], group_col, value_col, pairs)
        if not d_stats.empty:
            d_stats['DIV'] = div
            div_stats_list.append(d_stats)
    
    stats_df = pd.concat(div_stats_list) if div_stats_list else pd.DataFrame()
    agg_stats_df = run_welch_stats(sub, group_col, value_col, pairs)

    # 3. FIGURE 1: GROUPED BARS ACROSS DIVs + SIGNIFICANCE
    summary = sub.groupby(['DIV', group_col])[value_col].agg(['mean', lambda x: sem(x, nan_policy='omit')])
    summary.columns = ['mean', 'sem']
    summary = summary.reset_index()

    fig_div, ax = plt.subplots(figsize=(max(8, len(divs)*3), 7))
    indices = np.arange(len(divs)) * 2.0 # Extra spacing for brackets
    bar_width = 0.4

    y_max_global = sub[value_col].max()
    yr = y_max_global * 0.12 # Vertical spacing unit for brackets

    for i, nt in enumerate(order):
        nt_summary = summary[summary[group_col] == nt].set_index('DIV').reindex(divs).reset_index()
        x_off = indices + (i * bar_width)
        
        # Plot Bars
        ax.bar(x_off, nt_summary['mean'], width=bar_width, color=palette.get(nt, 'gray'), 
               alpha=0.6, edgecolor='black', label=nt.replace('CSB3 ', ''))
        ax.errorbar(x_off, nt_summary['mean'], yerr=nt_summary['sem'], fmt='none', c='black', capsize=3)
        
        # Plot Jittered points
        for j, div in enumerate(divs):
            pts = sub[(sub['DIV'] == div) & (sub[group_col] == nt)][value_col]
            jitter = (np.random.rand(len(pts)) - 0.5) * (bar_width * 0.4)
            ax.scatter(np.full(len(pts), x_off[j]) + jitter, pts, color=palette.get(nt, 'gray'), 
                       edgecolor='black', alpha=0.3, s=15, zorder=3)

    # Draw Per-DIV significance bars
    for j, div in enumerate(divs):
        div_x_base = indices[j]
        day_stats = stats_df[stats_df['DIV'] == div] if not stats_df.empty else pd.DataFrame()
        if day_stats.empty: continue
        
        # Comparison 1: Healthy vs E198K
        s1 = day_stats[(day_stats['G1'] == order[0]) & (day_stats['G2'] == order[1])]
        if not s1.empty and s1['sig'].values[0] != 'ns':
            y = y_max_global + yr
            x1, x2 = div_x_base, div_x_base + bar_width
            ax.plot([x1, x1, x2, x2], [y-(yr*0.2), y, y, y-(yr*0.2)], color='black', lw=1)
            ax.text((x1+x2)/2, y, s1['sig'].values[0], ha='center', va='bottom', fontsize=9, fontweight='bold')
            
        # Comparison 2: Healthy vs E200K
        s2 = day_stats[(day_stats['G1'] == order[0]) & (day_stats['G2'] == order[2])]
        if not s2.empty and s2['sig'].values[0] != 'ns':
            y = y_max_global + (yr * 2.5)
            x1, x2 = div_x_base, div_x_base + (2 * bar_width)
            ax.plot([x1, x1, x2, x2], [y-(yr*0.2), y, y, y-(yr*0.2)], color='black', lw=1)
            ax.text((x1+x2)/2, y, s2['sig'].values[0], ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xticks(indices + bar_width)
    ax.set_xticklabels([f"DIV {int(d)}" for d in divs], fontweight='bold')
    ax.set_ylabel(feature_label, fontweight='bold')
    ax.set_title(f"{feature_label} per DIV (Welch's T-test)", fontweight='bold', pad=25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, y_max_global + (yr * 6))
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(1,1))

    # 4. FIGURE 2: AGGREGATED PLOT
    agg_summary = sub.groupby(group_col)[value_col].agg(['mean', lambda x: sem(x, nan_policy='omit')])
    agg_summary.columns = ['mean', 'sem']
    agg_summary = agg_summary.reindex(order).reset_index()

    fig_agg, ax_agg = plt.subplots(figsize=(5, 6))
    for i, nt in enumerate(order):
        row = agg_summary.iloc[i]
        pts = sub[sub[group_col] == nt][value_col]
        ax_agg.bar(i, row['mean'], width=0.7, color=palette.get(nt, 'gray'), alpha=0.6, edgecolor='black')
        ax_agg.errorbar(i, row['mean'], yerr=row['sem'], fmt='none', c='black', capsize=5)
        ax_agg.scatter(np.full(len(pts), i) + (np.random.rand(len(pts))-0.5)*0.4, pts, 
                       color=palette.get(nt, 'gray'), edgecolor='black', alpha=0.3, s=20, zorder=3)

    # Brackets for Aggregated
    for i, (target_geno, y_lvl) in enumerate([(order[1], 1), (order[2], 2.5)]):
        sig_row = agg_stats_df[(agg_stats_df['G1'] == order[0]) & (agg_stats_df['G2'] == target_geno)]
        if not sig_row.empty and sig_row['sig'].values[0] != 'ns':
            y = y_max_global + (yr * y_lvl)
            x1, x2 = 0, order.index(target_geno)
            ax_agg.plot([x1, x1, x2, x2], [y-(yr*0.2), y, y, y-(yr*0.2)], color='black', lw=1.2)
            ax_agg.text((x1+x2)/2, y, sig_row['sig'].values[0], ha='center', va='bottom', fontweight='bold')

    ax_agg.set_xticks(range(len(order)))
    ax_agg.set_xticklabels([n.replace('CSB3 ', '') for n in order], fontweight='bold')
    ax_agg.set_ylabel(feature_label, fontweight='bold')
    ax_agg.set_title(f"{feature_label} (Aggregated)", fontweight='bold', pad=25)
    ax_agg.spines[['top', 'right']].set_visible(False)
    ax_agg.set_ylim(0, y_max_global + (yr * 6))

    # 5. FIGURE 3: STATS TABLE
    fig_tbl, ax_tbl = plt.subplots(figsize=(10, 8))
    ax_tbl.axis('off')
    if not stats_df.empty or not agg_stats_df.empty:
        agg_tbl = agg_stats_df.copy(); agg_tbl['DIV'] = 'AGGREGATED'
        full_stats = pd.concat([stats_df, agg_tbl])[['DIV', 'Comparison', 'test_used', 'p_raw', 'sig']]
        full_stats['p_raw'] = full_stats['p_raw'].apply(lambda x: f"{x:.4e}" if pd.notnull(x) else "N/A")
        
        table = ax_tbl.table(cellText=full_stats.values, colLabels=full_stats.columns, loc='center', cellLoc='center')
        table.auto_set_font_size(False); table.set_fontsize(8); table.scale(1.2, 1.2)
        ax_tbl.set_title(f"Statistical Summary Table: {feature_label}", fontweight='bold')

    plt.tight_layout()
    return fig_div, fig_agg, fig_tbl

def export_mea_analysis_to_pdf(df, cols, output_name="Report.pdf"):
    with PdfPages(output_name) as pdf:
        for col in cols:
            if col not in df.columns: continue
            print(f"Generating full analysis for: {col}")
            label = col.replace('nb_', '').replace('_', ' ').title()
            f1, f2, f3 = analyze_organoid_pairwise_pdf_version(df, col, feature_label=label)
            if f1 is not None:
                pdf.savefig(f1); pdf.savefig(f2); pdf.savefig(f3)
                plt.close(f1); plt.close(f2); plt.close(f3)
    print(f"\nPDF Report Saved: {output_name}")

# --- EXECUTION ---
# Using Welch's T-test for a robust biological comparison
export_mea_analysis_to_pdf(
    neuronal_assay_df, 
    ['nb_count', 'nb_rate_hz', 'nb_duration_mean_s', 'nb_spikes_per_burst_mean', 'num_units', 'bl_count', 'nb_burst_peak_mean', 'nb_peak_synchrony_mean'    ], 
    output_name='/mnt/Vol20tb1/user_workspaces/shruti/MEA_Analysis/MEA_Analysis_V2/MEA_Analysis/AnalyzedData/JGA_CSB3_010825_VD/MEA_Graphs_DIV19.pdf'
)

## Graphs ylim CUTOOF